# SWORD Reach Extraction for RivMapper v2

Batch-extracts SWORD v17b reaches and nodes for each river listed in the
v2 master CSV. Mirrors the v1 HydroRIVERS extraction pattern but operates
on SWORD topology and outputs geopackages instead of shapefiles.

## Workflow

1. **User identifies reaches in a GIS** (QGIS, ArcGIS, etc.) by loading the
   continental SWORD geopackage from `{working_directory}\SWORD\` and
   inspecting reach IDs along their river of interest.
2. **User fills the master CSV** with one row per river, providing
   `river_name`, `working_directory`, `sword_cont_abb`, `us_sword_reach_id`,
   and `ds_sword_reach_id`.
3. **Run this notebook** — it groups rivers by continent, loads each
   continental SWORD geopackage once, traces the path from US to DS via
   `rch_id_dn` topology, and writes per-river reach + node geopackages and
   a reach summary CSV.

## Outputs (per river)

```
{working_directory}\SWORD\Extracted_Rivers\{river_name}\
    ├── {river_name}_SWORD_reaches.gpkg
    ├── {river_name}_SWORD_reaches.csv
    └── {river_name}_SWORD_nodes.gpkg
```

The nodes geopackage carries SWORD's native attributes plus two derived
columns: `reach_order` (1 = upstream, N = downstream along the traced
path) and `node_order` (sequential index along the full US→DS path).

In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import requests
import zipfile
import hashlib
from tqdm.notebook import tqdm

In [2]:
# ── CONFIG ───────────────────────────────────────────────────────────
CONFIG = {
    "master_csv_path": Path(
        r"E:\Dissertation\Data\Template_RivMapper_master.csv"
    ),
    "sword_version": "v17b",
    "overwrite": False,  # if True, re-process rivers even if outputs exist
}

# ── Zenodo download constants ────────────────────────────────────────
ZENODO_GPKG_URL = "https://zenodo.org/records/15299138/files/SWORD_v17b_gpkg.zip"
ZENODO_MD5 = "fdaaad6f6b0b58f4212b99d2ad98188c"
VALID_CONTINENTS = {"af", "as", "eu", "na", "oc", "sa"}

print(f"Master CSV:    {CONFIG['master_csv_path']}")
print(f"SWORD version: {CONFIG['sword_version']}")

Master CSV:    E:\Dissertation\Data\Template_RivMapper_master.csv
SWORD version: v17b


In [3]:
def ensure_sword_downloaded(continent, sword_dir, version):
    """
    Return (reaches_path, nodes_path) for the requested continent.

    If both geopackages are already on disk, returns immediately.
    Otherwise downloads the full SWORD v17b zip from Zenodo, extracts
    all .gpkg files, and removes the zip.
    """
    reaches_path = sword_dir / f"{continent}_sword_reaches_{version}.gpkg"
    nodes_path = sword_dir / f"{continent}_sword_nodes_{version}.gpkg"

    if reaches_path.exists() and nodes_path.exists():
        return reaches_path, nodes_path

    print(f"  SWORD geopackages for '{continent}' not found in {sword_dir}.")
    print(f"  Downloading from Zenodo: {ZENODO_GPKG_URL}")
    zip_path = sword_dir / "SWORD_v17b_gpkg.zip"

    response = requests.get(ZENODO_GPKG_URL, stream=True)
    response.raise_for_status()
    total_size = int(response.headers.get("Content-Length", 0))

    with open(zip_path, "wb") as f, tqdm(
        total=total_size, unit="B", unit_scale=True, unit_divisor=1024,
        desc="Downloading SWORD",
    ) as pbar:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            pbar.update(len(chunk))

    # MD5 check (warn-only)
    md5 = hashlib.md5()
    with open(zip_path, "rb") as f:
        for block in iter(lambda: f.read(8192), b""):
            md5.update(block)
    if md5.hexdigest() != ZENODO_MD5:
        print(f"  WARNING: MD5 mismatch — expected {ZENODO_MD5}, got {md5.hexdigest()}")
    else:
        print("  MD5 checksum verified.")

    # Extract .gpkg files (skip macOS resource forks)
    print("  Extracting geopackage files...")
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.namelist():
            filename = Path(member).name
            if filename.endswith(".gpkg") and not filename.startswith("._"):
                target = sword_dir / filename
                with zf.open(member) as src, open(target, "wb") as dst:
                    dst.write(src.read())

    zip_path.unlink()
    print(f"  SWORD files ready for '{continent}'.")
    return reaches_path, nodes_path

In [4]:
def trace_reaches(us_reach_id, ds_reach_id, reaches_gdf, max_iter=1000):
    """
    Walk from `us_reach_id` downstream via SWORD's `rch_id_dn` topology
    until `ds_reach_id` is reached.

    Validates that the US reach is actually upstream of the DS reach by
    comparing `dist_out` (distance from outlet, meters) — US should have
    a larger value than DS.

    SWORD stores `rch_id_dn` as a string. When `n_rch_dn > 1` it's
    space-separated; when n_rch_dn=1 it's a single integer-looking
    string. Both forms parse correctly via `str.split()`. This function
    takes the first non-zero entry at each step for a single-channel walk.

    Returns
    -------
    list[int]
        Ordered reach IDs from upstream to downstream (inclusive).
    """
    dn_lookup = reaches_gdf.set_index("reach_id")["rch_id_dn"].to_dict()
    dist_lookup = reaches_gdf.set_index("reach_id")["dist_out"].to_dict()

    # ── Pre-trace validation: US must be upstream of DS ─────────────
    us_dist = dist_lookup.get(us_reach_id)
    ds_dist = dist_lookup.get(ds_reach_id)
    if us_dist is None:
        raise RuntimeError(
            f"us_sword_reach_id {us_reach_id} not found in reaches_gdf. "
            f"Check sword_cont_abb is correct."
        )
    if ds_dist is None:
        raise RuntimeError(
            f"ds_sword_reach_id {ds_reach_id} not found in reaches_gdf. "
            f"Check sword_cont_abb is correct."
        )
    if us_dist <= ds_dist:
        raise RuntimeError(
            f"us_sword_reach_id ({us_reach_id}, dist_out={us_dist:.0f} m) "
            f"is not upstream of ds_sword_reach_id ({ds_reach_id}, "
            f"dist_out={ds_dist:.0f} m). US dist_out must exceed DS "
            f"dist_out. Endpoints may be swapped."
        )

    # ── Walk downstream via rch_id_dn ───────────────────────────────
    path = [us_reach_id]
    current_id = us_reach_id

    for _ in range(max_iter):
        if current_id == ds_reach_id:
            return path

        dn_raw = dn_lookup.get(current_id)
        if dn_raw is None:
            raise RuntimeError(
                f"Reach {current_id} not found mid-walk. "
                f"Possible topology break."
            )

        dn_ids = [int(x) for x in str(dn_raw).split() if int(x) != 0]
        if not dn_ids:
            raise RuntimeError(
                f"Reach {current_id} has no downstream neighbour "
                f"(rch_id_dn='{dn_raw}'). May be the network outlet — "
                f"check your endpoint pair."
            )

        current_id = dn_ids[0]
        path.append(current_id)

    raise RuntimeError(
        f"Did not reach {ds_reach_id} within {max_iter} iterations. "
        f"Possible topology break or wrong endpoint pair."
    )

In [5]:
def extract_river(row, reaches_gdf, nodes_gdf, version, overwrite=False):
    """
    Extract SWORD reaches and nodes for one river, given the already-loaded
    continental reaches and nodes GeoDataFrames.

    The continent-level GeoDataFrames are loaded once by the caller and
    passed in for each river that shares a continent — avoiding repeated
    expensive geopackage reads.

    Returns one of: "ok (...)", "skipped", "failed: <reason>".
    """
    river_name = row["river_name"]
    us_id = int(row["us_sword_reach_id"])
    ds_id = int(row["ds_sword_reach_id"])
    working_directory = Path(row["working_directory"])

    # ── Set up output paths ─────────────────────────────────────────
    out_dir = working_directory / "SWORD" / "Extracted_Rivers" / river_name
    reaches_out = out_dir / f"{river_name}_SWORD_reaches.gpkg"
    csv_out = out_dir / f"{river_name}_SWORD_reaches.csv"
    nodes_out = out_dir / f"{river_name}_SWORD_nodes.gpkg"

    if (
        not overwrite
        and reaches_out.exists()
        and csv_out.exists()
        and nodes_out.exists()
    ):
        return "skipped"

    out_dir.mkdir(parents=True, exist_ok=True)

    # ── Trace path through SWORD topology ──────────────────────────
    try:
        path = trace_reaches(us_id, ds_id, reaches_gdf)
    except RuntimeError as e:
        return f"failed: {e}"

    # ── Extract traced reaches in US→DS order ───────────────────────
    reaches_subset = reaches_gdf[reaches_gdf["reach_id"].isin(path)].copy()
    order_map = {rid: i + 1 for i, rid in enumerate(path)}
    reaches_subset["reach_order"] = reaches_subset["reach_id"].map(order_map)
    reaches_subset = reaches_subset.sort_values("reach_order").reset_index(drop=True)

    reaches_subset.to_file(reaches_out, driver="GPKG")
    reaches_subset.drop(columns="geometry").to_csv(csv_out, index=False)

    # ── Filter nodes to traced reaches, add ordering ───────────────
    nodes_subset = nodes_gdf[nodes_gdf["reach_id"].isin(path)].copy()
    nodes_subset["reach_order"] = nodes_subset["reach_id"].map(order_map)
    # dist_out decreases downstream — descending sort within reach = US→DS.
    nodes_subset = nodes_subset.sort_values(
        ["reach_order", "dist_out"], ascending=[True, False]
    ).reset_index(drop=True)
    nodes_subset["node_order"] = range(1, len(nodes_subset) + 1)

    nodes_subset.to_file(nodes_out, driver="GPKG")

    return f"ok ({len(path)} reaches, {len(nodes_subset)} nodes)"

In [6]:
# ── Load and validate master CSV ────────────────────────────────────
master = pd.read_csv(CONFIG["master_csv_path"])

required_cols = {
    "river_name", "sword_cont_abb",
    "us_sword_reach_id", "ds_sword_reach_id",
    "working_directory",
}
missing = required_cols - set(master.columns)
if missing:
    raise ValueError(f"Master CSV missing required columns: {missing}")

# Normalize continent codes to lowercase and validate.
master["sword_cont_abb"] = master["sword_cont_abb"].str.lower()
invalid = set(master["sword_cont_abb"]) - VALID_CONTINENTS
if invalid:
    raise ValueError(
        f"Invalid sword_cont_abb values: {invalid}. "
        f"Must be one of {VALID_CONTINENTS}."
    )

# ── Scan unique continents and unique working directories ───────────
unique_continents = sorted(master["sword_cont_abb"].unique())
unique_working_dirs = sorted(master["working_directory"].unique())

print(f"Loaded {len(master)} rivers from master CSV.")
print(f"Unique continents to load: {unique_continents}")
print(f"Unique working directories: {len(unique_working_dirs)}")
print()

# ── Ensure SWORD is downloaded for every working directory ──────────
# Each unique working_directory needs its own SWORD/ subfolder with the
# continental geopackages. If two rivers share a working_directory, the
# check only runs once per continent there.
print("─── Verifying SWORD downloads ───")
for wd in unique_working_dirs:
    sword_dir = Path(wd) / "SWORD"
    sword_dir.mkdir(parents=True, exist_ok=True)
    continents_here = sorted(
        master.loc[master["working_directory"] == wd, "sword_cont_abb"].unique()
    )
    print(f"  {sword_dir}")
    print(f"    needs continents: {continents_here}")
    for cont in continents_here:
        ensure_sword_downloaded(cont, sword_dir, CONFIG["sword_version"])
print()

# ── Process rivers grouped by continent ─────────────────────────────
results = []
for continent in unique_continents:
    print(f"─── Processing continent: {continent.upper()} ───")
    rivers_in_continent = master[master["sword_cont_abb"] == continent]

    # All copies of a continental gpkg are byte-identical, so load from
    # whichever working_directory the first row in this group points at.
    source_wd = Path(rivers_in_continent.iloc[0]["working_directory"])
    source_sword_dir = source_wd / "SWORD"
    reaches_path = source_sword_dir / f"{continent}_sword_reaches_{CONFIG['sword_version']}.gpkg"
    nodes_path = source_sword_dir / f"{continent}_sword_nodes_{CONFIG['sword_version']}.gpkg"

    print(f"  Loading {reaches_path.name} ...", end=" ", flush=True)
    reaches_raw = gpd.read_file(reaches_path, engine="pyogrio")
    reaches_gdf = reaches_raw[reaches_raw["type"] == 1].copy()
    print(f"{len(reaches_gdf):,} river reaches.")

    print(f"  Loading {nodes_path.name} ...", end=" ", flush=True)
    nodes_gdf = gpd.read_file(nodes_path, engine="pyogrio")
    print(f"{len(nodes_gdf):,} nodes.")
    print()

    # Process each river in this continent.
    for _, row in rivers_in_continent.iterrows():
        river_name = row["river_name"]
        print(f"  → {river_name} ... ", end="", flush=True)
        try:
            status = extract_river(
                row,
                reaches_gdf=reaches_gdf,
                nodes_gdf=nodes_gdf,
                version=CONFIG["sword_version"],
                overwrite=CONFIG["overwrite"],
            )
        except Exception as e:
            status = f"failed: {type(e).__name__}: {e}"
        print(status)
        results.append({"river_name": river_name, "continent": continent, "status": status})

    # Free memory before loading the next continent.
    del reaches_raw, reaches_gdf, nodes_gdf
    print()

# ── Summary ─────────────────────────────────────────────────────────
summary = pd.DataFrame(results)
print("──── Summary ────")
print(summary.to_string(index=False))

Loaded 2 rivers from master CSV.
Unique continents to load: ['oc', 'sa']
Unique working directories: 1

─── Verifying SWORD downloads ───
  E:\Dissertation\Data\SWORD
    needs continents: ['oc', 'sa']

─── Processing continent: OC ───
  Loading oc_sword_reaches_v17b.gpkg ... 8,850 river reaches.
  Loading oc_sword_nodes_v17b.gpkg ... 686,595 nodes.

  → OkTedi ... ok (15 reaches, 869 nodes)

─── Processing continent: SA ───
  Loading sa_sword_reaches_v17b.gpkg ... 31,662 river reaches.
  Loading sa_sword_nodes_v17b.gpkg ... 2,135,054 nodes.

  → Beni ... ok (60 reaches, 4094 nodes)

──── Summary ────
river_name continent                      status
    OkTedi        oc  ok (15 reaches, 869 nodes)
      Beni        sa ok (60 reaches, 4094 nodes)


## Notes

- The first time a continent is needed at a given `working_directory`,
  the full SWORD v17b zip (~2 GB) downloads from Zenodo. Subsequent runs
  reuse the cached files.
- Rivers with all three output files already present are skipped unless
  `overwrite=True` in CONFIG.
- The path tracer assumes a single-channel walk via the first non-zero
  `rch_id_dn` entry. For braided/distributary reaches with multiple
  downstream neighbors, this may pick an unintended branch. If the
  traced path looks wrong, inspect intermediate `rch_id_dn` values in
  GIS and consider breaking the river into multiple sub-segments.
- Pre-trace validation compares `dist_out` of the US and DS endpoints.
  US must have a larger `dist_out` (further from outlet) than DS.
  Swapped endpoints fail fast with a clear message before any tracing.

## Next steps

- Graduate `trace_reaches()` and `ensure_sword_downloaded()` into
  `rivmapper_v2/sword_reaches.py` with unit tests.
- Consider handling braided/distributary topology with user prompts at
  each split, or a `branch_hint` column in the master CSV.